# Colab SSH Bootstrap

Run all cells to start an SSH tunnel into this Colab runtime.

**Order:** colab-ssh runs **before** the Google Drive mount so the `trycloudflare.com` hostname usually appears **before** the Drive consent dialog. That way browser automation (and you) can read the hostname without Drive OAuth blocking the first code cell.

Connect from Cursor after the hostname line appears:
```
scripts/connect_colab.sh <HOSTNAME>
```

The Drive cell is for a persistent clone under My Drive. If you skip Drive access, the next cell uses `/content/recsys_playground` (not persisted across sessions).

In [ ]:
!pip install colab-ssh --upgrade -q
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="cursorssh")

# Relay hostname to agent via ntfy.sh (automated pickup)
from colab_ssh.get_tunnel_config import get_argo_tunnel_config
import urllib.request
_info = get_argo_tunnel_config()
urllib.request.urlopen(urllib.request.Request(
    'https://ntfy.sh/colab-ssh-allyoushawn-e3b76118-69fd-4a29-89c1-c34e07b0977e',
    data=_info['domain'].encode(), method='POST'
))
print(f"Hostname relayed: {_info['domain']}")

In [ ]:
# Mount Google Drive for persistent storage (runs after SSH so hostname is available first)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.isdir('/content/drive/MyDrive'):
    WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
else:
    WORK_DIR = '/content/recsys_playground'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'

if not os.path.exists(repo_dir):
    !git clone {repo_url}

%cd {repo_dir}
!git pull origin main
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn requests papermill